In [ ]:
!pip install transformers scikit-learn pandas seaborn matplotlib

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("Twitter_Data.csv")

# Rename columns
df.rename(columns={'clean_text':'text', 'category':'label'}, inplace=True)

# Remove missing values
df = df.dropna()

# Remove neutral (0)
df = df[df['label'] != 0]

# Convert labels (-1 → 0, 1 → 1)
df['label'] = df['label'].map({-1:0, 1:1})

# Convert to integer
df['label'] = df['label'].astype(int)

#  Reduce dataset (IMPORTANT for speed)
df = df.sample(3000, random_state=42)

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(list(train_texts), padding=True, truncation=True)
val_encodings = tokenizer(list(val_texts), padding=True, truncation=True)
test_encodings = tokenizer(list(test_texts), padding=True, truncation=True)

In [ ]:
import torch

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)
test_dataset = Dataset(test_encodings, test_labels)

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(p):
    preds = p.predictions.argmax(axis=1)
    labels = p.label_ids

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate(test_dataset)
print(results)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)

cm = confusion_matrix(test_labels, preds)

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

**Conclusion:**

The BERT model achieved strong performance on sentiment classification
The model shows high accuracy (~87%) indicating good prediction capability
Precision and recall values indicate balanced classification performance
The confusion matrix shows most predictions are correct
The model performs better in predicting positive sentiments